# This is for extracting collapsed faults of each gate 

In [2]:
import os

In [3]:
benchmark_name = "c1908_netlist"
input_transition = 0.01 #used in generation of sysC TB

In [4]:
## verilog preprocess:
file_path = benchmark_name+".v"
preVer = []
with open(file_path, "r") as file:
    for line in file:
        if "not" in line:
            outp = (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" ")
            inp = (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" ")
            line = line.replace("not ", "nand ").replace("NOT1_","NAND2_rep").replace(inp+");", inp+", "+inp+");")
            preVer.append(line)
        else:
            preVer.append(line)

directory_path = benchmark_name+"/Verilog/"
os.makedirs(directory_path, exist_ok=True)
Ver_path = os.path.join(directory_path+benchmark_name+".v")
with open(Ver_path, "w") as file:
    file.writelines(preVer)
file.close()


In [5]:
## One input gates
class gate1:
    def __init__(self, name, numOfInp, outputZN, inputA1, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.outputZN = outputZN
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n ZN = {self.outputZN}\n gate Type = {self.type}"


In [6]:
## Two input gates
class gate2:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.outputZN = outputZN
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n ZN = {self.outputZN}\n gate Type = {self.type}"


In [7]:
## Three input gates
class gate3:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, inputA3, gateNum):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.inputA3 = inputA3
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n A3 = {self.inputA3}\n ZN = {self.outputZN}"


In [8]:
## Four input gates
class gate4:
    def __init__(self, name, numOfInp, outputZN, inputA1, inputA2, inputA3, inputA4, gateNum):
        self.name = name
        self.gateNum = gateNum
        self.numOfInp = numOfInp
        self.inputA1 = inputA1
        self.inputA2 = inputA2
        self.inputA3 = inputA3
        self.inputA4 = inputA4
        self.outputZN = outputZN
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n A1 = {self.inputA1}\n A2 = {self.inputA2}\n A3 = {self.inputA3}\n A4 = {self.inputA4}\n ZN = {self.outputZN}"


In [9]:
directory_path = benchmark_name+"/Verilog/"
file_path = os.path.join(directory_path+benchmark_name+".v")
i = 0
j = 0
net = []
required_gates = []
newGate = False
PIs = []
POs = []
assign_left = []
assign_right = []

numOfPI = 0
numOfPO = 0

wires=[]
wire_lines=""
with open(file_path, "r") as file:
    for line in file:
        if 'wire' in line:
            if(';' in line):
                wire_lines += line.rstrip('\n').split('wire')[1]
            else:
                wire_lines += line.rstrip('\n').split('wire')[1]
                while not(';' in line):
                    line = file.readline()  
                    wire_lines += line.rstrip('\n')
        wires = wire_lines
wires = wires.replace(" ", "").replace(";",",").split(',')
if (wires[len(wires)-1] == ""):
    wires = wires[:-1]
else: 
    wires[len(wires)-1] = wires[len(wires)-1].strip(',')
numOfwire = len(wires)
file.close()


with open(file_path, "r") as file:
    for line in file:
        if 'input' in line:
            PI= line.rstrip('\n').split('input')[1].strip(';').replace(" ", "")
            numOfPI += 1
            # print(numOfPI)
            PIs.append(PI)
            # PI_lines = line.rstrip('\n').split('input')[1]
            # ## handling enters
            # while ';' not in line:
            #     line = file.readline()
            # numOfPI = len(PI_lines.split()[1:])
            # PIs = PI_lines.replace(" ", "").strip(';').split(',')

        elif 'output' in line:
            PO= line.rstrip('\n').split('output')[1].strip(';').replace(" ", "")
            numOfPO += 1
            # print(numOfPO)
            POs.append(PO)
            # PO_lines = line.rstrip('\n').split('output')[1]
            # while ';' not in line:
            #     line = file.readline()
            #     PO_lines += line.rstrip('\n')
            # numOfPO = len(PO_lines.split()[1:])
            # POs = PO_lines.replace(" ", "").strip(';').split(',')
            
        
        elif 'NAND' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            # print(line.split())
            currGateName = line.split()[1]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("NAND" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("NAND")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "NAND"+str(numOfGateInput)+"_X1"
            # if (numOfGateInput == 1):
            #     net.append(gate1(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 2):
            #     net.append(gate2(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # if (numOfGateInput == 3):
            #     net.append(gate3(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
        
        elif 'NOR' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            currGateName = line.split()[1]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("NOR" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("NOR")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "NOR"+str(numOfGateInput)+"_X1"
            # newGate = True
            # i=i+1
            # currGateName = line.split()[1][0:4]
            # if (line.split()[1][0:4] not in required_gates):
            #     required_gates.append(line.split()[1][0:4])
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            # # if (numOfGateInput == 2):
            # #     net.append(gate2(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
            # # if (numOfGateInput == 3):
            # #     net.append(gate3(currGateName, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
        
        elif 'NOT' in line:
            gateInps =[]
            # print(line)
            newGate = True
            numOfGateInput = 0
            i=i+1
            currGateName = line.split()[1]
            # print(currGateName)
            # print(line.split()[1][0:5])
            if ("INV" not in required_gates):
                # required_gates.append(line.split()[1][0:5])
                required_gates.append("INV")
            line = file.readline() ##first input
            gateInps.append((line.split('(')[1].split(')')[0]))
            numOfGateInput += 1
            line = file.readline() ##first input
            while ',' in line:
                gateInps.append((line.split('(')[1].split(')')[0]))
                numOfGateInput += 1
                line = file.readline() ##first input
            # print(gateInps)
            # print(line)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
            numOfGateInput = numOfGateInput
            # print(numOfGateInput)
            gateOutp = (line.split('(')[1].split(')')[0])
            # print(gateOutp)
            gateType = "INV"+str(numOfGateInput)+"_X1"
            # newGate = True
            # i=i+1
            # currGateName = line.split()[1][0:4]
            # currGateName = "INV"
            # if (currGateName not in required_gates):
            #     required_gates.append(currGateName)
            # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1

        elif 'assign' in line:
            # print(line.split())
            assign_left.append(line.split()[1])
            assign_right.append(line.split()[3].strip(';'))

        # elif ' or' in line:
        #     # print(line)
        #     newGate = True
        #     i=i+1
        #     numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
        #     currGateName = "OR"+str(numOfGateInput)
        #     # currGateName = line.split()[1][0:3]
        #     if (currGateName not in required_gates):
        #         required_gates.append(currGateName)
        #     # numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1


        # elif 'and' in line:
        #     newGate = True
        #     i=i+1
        #     numOfGateInput = len(line[line.index('(')+1:line.index(')')].split(',')) -1
        #     currGateName = "AND"+str(numOfGateInput)
        #     if (currGateName not in required_gates):
        #         required_gates.append(currGateName)
        else:
            newGate = False

        if (newGate):
            if (numOfGateInput == 1):
                # net.append(gate1(currGateName, 1, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), i))
                net.append(gate1(currGateName, 1, gateOutp, gateInps[0], i, gateType))

            elif (numOfGateInput == 2):
                # net.append(gate2(currGateName, 2,(line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), i))
                net.append(gate2(currGateName, 2, gateOutp, gateInps[0], gateInps[1], i, gateType))

            elif (numOfGateInput == 3):
                net.append(gate3(currGateName, 3, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), i))
            elif (numOfGateInput == 4):
                net.append(gate4(currGateName, 4, (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[2].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[3].strip(" "), (line[line.index('(')+1:line.index(')')].split(','))[4].strip(" "), i))
file.close()

In [10]:
print(assign_left)
print(assign_right)


['N1001', 'N1002', 'N1003', 'N1004', 'N1005', 'N1006', 'N1007', 'N1008', 'N1009', 'N1148', 'N1149', 'N1151', 'N1152', 'N1153', 'N1154', 'N1156', 'N1161', 'N1205', 'N1207', 'N1209', 'N1211', 'N1213', 'N1215', 'N1217', 'N1219', 'N1220', 'N1222', 'N1223', 'N1225', 'N1228', 'N1238', 'N1240', 'N1241', 'N257', 'N260', 'N283', 'N297', 'N303', 'N316', 'N326', 'N331', 'N343', 'N346', 'N349', 'N352', 'N355', 'N358', 'N361', 'N364', 'N367', 'N370', 'N373', 'N376', 'N379', 'N382', 'N385', 'N388', 'N888', 'N889', 'N890', 'N891', 'N892', 'N894', 'N895', 'N913', 'N914', 'N915', 'N916', 'N917', 'N918', 'N919', 'N920', 'N938', 'N942', 'N946', 'N950', 'N954', 'N958', 'N968', 'N972', 'N976', 'N980', 'N984', 'N988', 'N989', 'N990', 'N992', 'N993', 'N997', '_0231_', '_0358_', '_0575_', '_0668_', '_0818_', '_0232_', '_0244_', '_0280_', '_0305_', '_0331_', '_0369_', '_0380_', '_0412_', '_0424_', '_0450_', '_0483_', '_0494_', '_0555_', '_0603_', '_0622_', '_0649_', '_0659_', '_0678_', '_0707_', '_0727_', '_07

In [11]:
print(net[3])

 gateNum = 4
 gateName = _0843_
 A1 = _0786_
 A2 = _0767_
 ZN = _0787_
 gate Type = NOR2_X1


In [12]:
# num of gates
len(net)

576

In [13]:
required_gates

['INV', 'NOR', 'NAND']

In [14]:
print(len(PIs))
print(len(POs))

33
25


In [15]:
POs

['N2753',
 'N2754',
 'N2755',
 'N2756',
 'N2762',
 'N2767',
 'N2768',
 'N2779',
 'N2780',
 'N2781',
 'N2782',
 'N2783',
 'N2784',
 'N2785',
 'N2786',
 'N2787',
 'N2811',
 'N2886',
 'N2887',
 'N2888',
 'N2889',
 'N2890',
 'N2891',
 'N2892',
 'N2899']

In [16]:
print(net[2])

 gateNum = 3
 gateName = _0842_
 A1 = _0622_
 ZN = _0786_
 gate Type = INV1_X1


In [30]:
# read the faults list, store the content in a dectionary, key: net, value: SA
directory_path = "c1908_faultlist.flt"
# file_path = os.path.join(directory_path+benchmark_name+".v")
faultDict = {}
file_path = directory_path
with open(file_path, "r") as file:
    for line in file:  
        wire, SA = line.split()
        print(wire)
        print(SA)      
        if ( wire in faultDict):
            SAs=[]
            SAs.append(faultDict[wire])
            SAs.append(SA)
            faultDict[wire] = SAs
        else:
            faultDict[wire] = SA.strip("\n")

print(faultDict)

N1
s@0
N1
s@1
N4
s@0
N4
s@1
N7
s@0
N7
s@1
N10
s@0
N10
s@1
N13
s@0
N13
s@1
N16
s@0
N16
s@1
N19
s@0
N19
s@1
N22
s@0
N22
s@1
N25
s@0
N25
s@1
N28
s@0
N28
s@1
N31
s@0
N31
s@1
N34
s@0
N34
s@1
N37
s@0
N37
s@1
N40
s@0
N40
s@1
N43
s@0
N43
s@1
N46
s@0
N46
s@1
N49
s@0
N49
s@1
N53
s@0
N53
s@1
_0857_.Y
s@1
_0857_.Y
s@0
_1022_.Y
s@1
_1022_.Y
s@0
N63
s@0
N63
s@1
N66
s@0
N66
s@1
_0842_.Y
s@1
_0842_.Y
s@0
N72
s@0
N72
s@1
N76
s@0
N76
s@1
N79
s@0
N79
s@1
N82
s@0
N82
s@1
N85
s@0
N85
s@1
N88
s@0
N88
s@1
N91
s@0
N91
s@1
N94
s@0
N94
s@1
N99
s@0
N99
s@1
N104
s@0
N104
s@1
_0840_.Y
s@0
_0844_.A
s@0
_0845_.Y
s@0
_0845_.Y
s@1
_0841_.Y
s@0
_0843_.A
s@0
_0843_.Y
s@1
_0843_.Y
s@0
_0846_.Y
s@0
_0846_.B
s@0
_0846_.A
s@0
_0846_.Y
s@1
_0847_.Y
s@0
_0847_.Y
s@1
_0848_.Y
s@0
_0848_.B
s@0
_0848_.A
s@0
_0848_.Y
s@1
_0849_.Y
s@0
_0854_.B
s@0
_0854_.Y
s@1
_0853_.A
s@0
_0854_.Y
s@0
_0853_.Y
s@0
_0850_.Y
s@0
_0850_.Y
s@1
_0852_.Y
s@1
_0851_.B
s@0
_0851_.A
s@0
_0852_.Y
s@0
_0855_.Y
s@0
_0855_.Y
s@1
_0856_.Y
s@0
_0856_.Y
s@1
_085

In [31]:
# del my_dict['b']
newFaultDict = {}
POcount=0
PIcount=0
count =0 
for item in faultDict:
    add = True
    if (item  in POs):
        for g in net:
            if(g.outputZN == assign_right[assign_left.index(item)]):
                new_element_key = g.name+".Y"
                new_element_value = faultDict[item]
                newFaultDict[new_element_key] = new_element_value
                POcount+=1
                # count+=len(new_element_value)
    else:
        gateName = item.split('.')[0]
        for g in net:
            if(g.name == gateName and g.numOfInp==2): ## if it is a two input gate
                if ((g.inputA1 in assign_left)):
                    if(assign_right[assign_left.index(g.inputA1)] in PIs):
                        print("cought PI")
                        print(gateName)
                        print(g.inputA1)
                        add = False
                        # break
                if((g.inputA2 in assign_left)):
                    if(assign_right[assign_left.index(g.inputA2)] in PIs):
                        print("cought PI")
                        print(gateName)
                        print(g.inputA1)
                        add = False
                        # break
            if(g.name == gateName and g.numOfInp==1):
                if ((g.inputA1 in assign_left)):
                    if(assign_right[assign_left.index(g.inputA1)] in PIs):
                        print("cought PI")
                        print(gateName)
                        print(g.inputA1)
                        add = False
                        # break
        if(add):
            newFaultDict[item] = faultDict[item]
            # count+=len(faultDict[item])


cought PI
_0857_
_0814_
cought PI
_1022_
_0603_
cought PI
_0842_
_0622_
cought PI
_0840_
_0259_
cought PI
_0841_
_0668_
cought PI
_0846_
_0801_
cought PI
_0846_
_0801_
cought PI
_0846_
_0801_
cought PI
_0847_
_0305_
cought PI
_0848_
_0254_
cought PI
_0848_
_0254_
cought PI
_0848_
_0254_
cought PI
_0850_
_0575_
cought PI
_0856_
_0234_
cought PI
_0858_
_0786_
cought PI
_0858_
_0786_
cought PI
_0861_
_0818_
cought PI
_0862_
_0244_
cought PI
_0864_
_0821_
cought PI
_0864_
_0821_
cought PI
_0864_
_0821_
cought PI
_0862_
_0244_
cought PI
_0862_
_0244_
cought PI
_0863_
_0244_
cought PI
_0870_
_0369_
cought PI
_0871_
_0424_
cought PI
_0873_
_0424_
cought PI
_0873_
_0424_
cought PI
_0873_
_0424_
cought PI
_0873_
_0424_
cought PI
_0873_
_0424_
cought PI
_0873_
_0424_
cought PI
_0876_
_0483_
cought PI
_0877_
_0659_
cought PI
_0879_
_0837_
cought PI
_0879_
_0837_
cought PI
_0879_
_0837_
cought PI
_0877_
_0659_
cought PI
_0877_
_0659_
cought PI
_0878_
_0659_
cought PI
_0897_
_0251_
cought PI
_0897_

In [32]:
newFaultDict

{'N1': ['s@0', 's@1'],
 'N4': ['s@0', 's@1'],
 'N7': ['s@0', 's@1'],
 'N10': ['s@0', 's@1'],
 'N13': ['s@0', 's@1'],
 'N16': ['s@0', 's@1'],
 'N19': ['s@0', 's@1'],
 'N22': ['s@0', 's@1'],
 'N25': ['s@0', 's@1'],
 'N28': ['s@0', 's@1'],
 'N31': ['s@0', 's@1'],
 'N34': ['s@0', 's@1'],
 'N37': ['s@0', 's@1'],
 'N40': ['s@0', 's@1'],
 'N43': ['s@0', 's@1'],
 'N46': ['s@0', 's@1'],
 'N49': ['s@0', 's@1'],
 'N53': ['s@0', 's@1'],
 'N63': ['s@0', 's@1'],
 'N66': ['s@0', 's@1'],
 'N72': ['s@0', 's@1'],
 'N76': ['s@0', 's@1'],
 'N79': ['s@0', 's@1'],
 'N82': ['s@0', 's@1'],
 'N85': ['s@0', 's@1'],
 'N88': ['s@0', 's@1'],
 'N91': ['s@0', 's@1'],
 'N94': ['s@0', 's@1'],
 'N99': ['s@0', 's@1'],
 'N104': ['s@0', 's@1'],
 '_0844_.A': 's@0',
 '_0845_.Y': ['s@0', 's@1'],
 '_0843_.A': 's@0',
 '_0843_.Y': ['s@1', 's@0'],
 '_0849_.Y': 's@0',
 '_0854_.B': 's@0',
 '_0854_.Y': ['s@1', 's@0'],
 '_0853_.A': 's@0',
 '_0853_.Y': 's@0',
 '_0852_.Y': ['s@1', 's@0'],
 '_0851_.B': 's@0',
 '_0851_.A': 's@0',
 '_085

In [33]:
gateFault_dict = {}
line = []
tot =0
for g in net:
    faults = ""
    print("new gate "+g.name)
    key = g.name

    if (g.name+".Y" in newFaultDict):
        if ("s@0" in newFaultDict[g.name+".Y"]):
            line.append("s@0 "+g.name+".Y\n")
            faults +="1"
            tot+=1
        else:
            faults +="0"
        if ("s@1" in newFaultDict[g.name+".Y"]):
            line.append("s@1 "+g.name+".Y\n")
            faults +="1"
            tot+=1

        else:
            faults +="0"
    else:
         faults +="00"

    if (g.name+".A" in newFaultDict):
    
        if ("s@0" in newFaultDict[g.name+".A"]):
            line.append("s@0 "+g.name+".A\n")
            faults +="1"
            tot+=1
        else:
            faults +="0"
        if ("s@1" in newFaultDict[g.name+".A"]):
            line.append("s@1 "+g.name+".A\n")
            faults +="1"
            tot+=1
        else:
            faults +="0"
    else:
         faults +="00"

    if (g.name+".B" in newFaultDict):
        if ("s@0" in newFaultDict[g.name+".B"]):
            line.append("s@0 "+g.name+".B\n")
            faults +="1"
            tot+=1 
        else:
            faults +="0"
        if ("s@1" in newFaultDict[g.name+".B"]):
            line.append("s@1 "+g.name+".B\n")
            faults +="1"
            tot+=1
        else:
            faults +="0"
    else:
         faults +="00"
         
    if (faults == ""):
         faults = "0000"
         print("DAMN U"+g.name)
         
    print(key)
    if(int(faults, 2) != 0):
        gateFault_dict[key] = int(faults, 2)
    if(key == "_450_"):
                print(gateFault_dict[key])
    # print(faults)
        

    # print(g.inputA2)


new gate _0840_
_0840_
new gate _0841_
_0841_
new gate _0842_
_0842_
new gate _0843_
_0843_
new gate _0844_
_0844_
new gate _0845_
_0845_
new gate _0846_
_0846_
new gate _0847_
_0847_
new gate _0848_
_0848_
new gate _0849_
_0849_
new gate _0850_
_0850_
new gate _0851_
_0851_
new gate _0852_
_0852_
new gate _0853_
_0853_
new gate _0854_
_0854_
new gate _0855_
_0855_
new gate _0856_
_0856_
new gate _0857_
_0857_
new gate _0858_
_0858_
new gate _0859_
_0859_
new gate _0860_
_0860_
new gate _0861_
_0861_
new gate _0862_
_0862_
new gate _0863_
_0863_
new gate _0864_
_0864_
new gate _0865_
_0865_
new gate _0866_
_0866_
new gate _0867_
_0867_
new gate _0868_
_0868_
new gate _0869_
_0869_
new gate _0870_
_0870_
new gate _0871_
_0871_
new gate _0872_
_0872_
new gate _0873_
_0873_
new gate _0874_
_0874_
new gate _0875_
_0875_
new gate _0876_
_0876_
new gate _0877_
_0877_
new gate _0878_
_0878_
new gate _0879_
_0879_
new gate _0880_
_0880_
new gate _0881_
_0881_
new gate _0882_
_0882_
new gate _0

In [34]:
tot

1190

In [35]:
gateFault_dict

{'_0843_': 56,
 '_0844_': 8,
 '_0845_': 48,
 '_0849_': 32,
 '_0851_': 10,
 '_0852_': 48,
 '_0853_': 40,
 '_0854_': 50,
 '_0855_': 48,
 '_0859_': 48,
 '_0860_': 58,
 '_0865_': 48,
 '_0866_': 58,
 '_0867_': 53,
 '_0868_': 32,
 '_0869_': 50,
 '_0872_': 42,
 '_0874_': 48,
 '_0875_': 32,
 '_0880_': 48,
 '_0881_': 32,
 '_0882_': 32,
 '_0883_': 42,
 '_0884_': 48,
 '_0885_': 42,
 '_0886_': 16,
 '_0887_': 52,
 '_0888_': 48,
 '_0889_': 42,
 '_0890_': 48,
 '_0891_': 53,
 '_0892_': 42,
 '_0893_': 21,
 '_0894_': 21,
 '_0895_': 32,
 '_0898_': 48,
 '_0902_': 42,
 '_0903_': 48,
 '_0904_': 32,
 '_0905_': 34,
 '_0907_': 48,
 '_0913_': 48,
 '_0914_': 42,
 '_0917_': 48,
 '_0919_': 48,
 '_0922_': 16,
 '_0923_': 49,
 '_0926_': 48,
 '_0927_': 21,
 '_0928_': 32,
 '_0929_': 56,
 '_0930_': 21,
 '_0931_': 48,
 '_0932_': 21,
 '_0934_': 21,
 '_0935_': 48,
 '_0936_': 42,
 '_0937_': 42,
 '_0938_': 48,
 '_0939_': 21,
 '_0940_': 48,
 '_0941_': 42,
 '_0942_': 48,
 '_0943_': 42,
 '_0944_': 42,
 '_0945_': 48,
 '_0946_': 

In [36]:
directory_path = benchmark_name+"/faults/"
os.makedirs(directory_path, exist_ok=True)
sysC_output_file_path = os.path.join(directory_path+"faultlist.flt")
with open(sysC_output_file_path, "w") as file:
    file.writelines(line)
file.close()

In [37]:
## systemC output
sysC_H = []
# for required_gate in required_gates:
#     sysC_H.append("#include \""+required_gate+"_X1.h\"\n")
sysC_H.append("#include \"Complex_NAgate_45.h\"\n")

sysC_H.append("SC_MODULE("+benchmark_name+")\n{\n")
sysC_H.append("\tsc_in <sc_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_H.append(PIs[PI_idx]+", ")
sysC_H.append(PIs[len(PIs)-1]+";\n")

sysC_H.append("\tsc_out <sc_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_H.append(POs[PO_idx]+", ")
sysC_H.append(POs[len(POs)-1]+";\n")

sysC_H.append("\tsc_signal <sc_logic> ")
for wire_idx in range(len(wires)-1):
    if((not(wires[wire_idx] in PIs)) and (not(wires[wire_idx] in POs))):
        sysC_H.append(wires[wire_idx]+", ")
sysC_H.append(wires[len(wires)-1]+";\n")

sysC_H.append("\tsc_in<sc_logic> endSim; \n")
sysC_H.append("\tsc_in<sc_logic> newTV; \n")
sysC_H.append("\tsc_uint<32> counter; \n")

sysC_H.append("\n\tint numOfGates;\n\tint totalObservedCombs;\n\tdouble GIC_Coverage;\n\n")


# sysC_H.append("\n\tint numOfGates;\n\tdouble t;\n\tdouble outLoad;\n\n")

i = 1
for g in net:
    # sysC_H.append("\t"+g.name+"_X1* "+g.name+"_Gate"+str(i)+";\n")
    sysC_H.append("\t"+g.type+"* "+g.name+"_Gate"+str(i)+";\n")
    i = i+1

sysC_H.append("\n\tSC_CTOR("+benchmark_name+")\n\t{\n\t\tnumOfGates = "+str(len(net))+";\n\n")

i = 1
for g in net:
    # sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+" = new "+g.name+"_X1(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")
    sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+" = new "+g.type+"(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")
    sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->id = "+str(i)+";\n")

    if (g.numOfInp == 1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A"+"("+g.inputA1+");\n")
    if (g.numOfInp == 2):
        if (g.name.startswith("XOR")): ## XOR has only 2
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->B"+"("+g.inputA2+");\n")
        else:    
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
    if (g.numOfInp == 3):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
    if (g.numOfInp == 4):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A4"+"("+g.inputA4+");\n")
    sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->ZN"+"("+g.outputZN+");\n")
    if(g.name in gateFault_dict):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->faults"+"="+str(gateFault_dict[g.name])+";\n")
    else:
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->faults"+"=0;\n")
    sysC_H.append("\t\tmodule_map["+str(i)+"] = "+g.name+"_Gate"+str(i)+";\n\n")

        


    # if (g.outputZN in POs):
    #     sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c"+" = outLoad;\n")
    # else:
    #     sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c = 0;\n")
    # sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->aged_time = t;\n\n")
    i = i+1

sysC_H.append("\t\tcout << \"all gates are instantiated \" << numOfGates << \"\\n\";\n")

sysC_H.append("\t\tSC_THREAD(assignments);\n")
# //sensitivity
j=0
unique_sens_list = list(set(assign_right))
sysC_H.append("\t\tsensitive")
for j in range(len(unique_sens_list)):
    sysC_H.append(" << "+unique_sens_list[j])
sysC_H.append(";\n")
sysC_H.append("\t\tSC_METHOD(GIC_Coverage_Calculator);\n")
sysC_H.append("\t\tsensitive << endSim<< newTV;\n\n\t}\n")

sysC_H.append("\tvoid ini();\n\tvoid assignments();\n\tvoid GIC_Coverage_Calculator();\n")

# j = 1
# for required_gate_idx in range(len(required_gates)):
#     sysC_H.append("\tvoid notifyAlfaCalc"+required_gates[required_gate_idx]+"(")
#     sysC_H.append(required_gates[required_gate_idx]+"_X1 *gate"+");\n")
#     # sysC_H.append(required_gates[len(required_gates)-1]+"_X1 *gate"+str(j)+");\n};\n")
#     j=j+1
sysC_H.append("\n};\n")


directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_output_file_path = os.path.join(directory_path+"netlist.h")
with open(sysC_output_file_path, "w") as file:
    file.writelines(sysC_H)
file.close()

In [29]:
sysC_C = []
sysC_C.append("#include \"netlist.h\"\n#include <cmath>\n")
sysC_C.append("std::ofstream GIC_logFile(\"GIC_logFile.txt\");\n\n")
sysC_C.append("void "+benchmark_name+"::assignments()\n{\n\twhile (true)\n\t{\n")

m=0
for m in range(len(assign_left)):
    sysC_C.append("\t\t"+assign_left[m]+".write("+assign_right[m]+");\n")

sysC_C.append("\n\t\twait();\n\t}\n}\n\n")

sysC_C.append("void "+benchmark_name+"::GIC_Coverage_Calculator()\n{\n")
sysC_C.append("\ttotalObservedCombs =\n")
n=0
for n in range(len(net)):
    if n==len(net)-1:
        sysC_C.append("\t"+net[n].name+"_Gate"+str(n+1)+"->numOfObservedCombs;\n")
    else:
        sysC_C.append("\t"+net[n].name+"_Gate"+str(n+1)+"->numOfObservedCombs +\n")

sysC_C.append("\tcout << \"END OF SIM=> Total Observed Combs = \" << totalObservedCombs << \"\\n\";\n")
sysC_C.append("\tcout << \"GIC Coverage = \" << GIC_Coverage << \"\\n\";\n\n}")



directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_C_output_file_path = os.path.join(directory_path+"netlist.cpp")
with open(sysC_C_output_file_path, "w") as file:
    file.writelines(sysC_C)
file.close()

In [17]:

sysC_TB_H = []
sysC_TB_H.append("#include \"netlist.h\"\n#include <fstream>\n\nSC_MODULE("+benchmark_name+"_TB)\n{\n\n")
sysC_TB_H.append("\tsc_signal <sc_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_TB_H.append("testData"+str(PI_idx+1)+", ")
sysC_TB_H.append("testData"+str(len(PIs))+";\n")

sysC_TB_H.append("\tsc_signal <sc_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_TB_H.append("testRes"+str(PO_idx+1)+", ")
sysC_TB_H.append("testRes"+str(len(POs))+";\n")

sysC_TB_H.append("\n\tsc_signal<sc_logic> reset, clock, end, newTV;\n\t"+benchmark_name+"* UUT;\n\n")
sysC_TB_H.append("\tstd::vector<std::string> testVecs;\n")

sysC_TB_H.append("\tSC_CTOR("+benchmark_name+"_TB)\n\t{\n\n\t\ttestVecs = read_testPtr (\"testPatterns.txt\");\n\t\tUUT = new "+benchmark_name+"(\""+benchmark_name+"_instance\");\n")
p=1
for PI in PIs:
    sysC_TB_H.append("\t\tUUT->"+PI.strip(" ")+"(testData"+str(p)+");\n")
    p=p+1

o=1
for PO in POs:
    sysC_TB_H.append("\t\tUUT->"+PO.strip(" ")+"(testRes"+str(o)+");\n")
    o=o+1

sysC_TB_H.append("\t\tUUT->endSim(end);\n")
sysC_TB_H.append("\t\tUUT->newTV(newTV);\n")
sysC_TB_H.append("\n\t\tSC_THREAD(testPtr);\n\t\tSC_THREAD(endOfSim);\n\t}\n")

sysC_TB_H.append("\n\tvoid endOfSim();\n\tvoid testPtr();\n\tstd::vector<std::string> read_testPtr(std::string filename);\n")
sysC_TB_H.append("};\n")




directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_H_output_file_path = os.path.join(directory_path+"TB.h")
with open(sysC_TB_H_output_file_path, "w") as file:
    file.writelines(sysC_TB_H)
file.close()

In [18]:
sysC_TB_C = []
sysC_TB_C.append("#include \"TB.h\"\n\n")
sysC_TB_C.append("void "+benchmark_name+"_TB::testPtr()\n{\n\twhile (true)\n\t{\n")

m=1
for PI in PIs:
    # sysC_TB_C.append("void powerGatesNetlistTB::testData"+str(m)+"Waveform()\n{\n\twhile (true)\n{\n")
    sysC_TB_C.append("\t\ttestData"+str(m)+".write(SC_LOGIC_0);\n")
    m=m+1

sysC_TB_C.append("\t\twait(1000, SC_NS);\n")
sysC_TB_C.append("\t\tfor (int testVecidx = 0; testVecidx < testVecs.size(); testVecidx++)\n\t\t\t{\n")
s=0
for s in range(len(PIs)):
    sysC_TB_C.append("\t\t\tif (testVecs[testVecidx]["+str(s)+"] == '0')\n\t\t\t\ttestData"+str(s+1)+".write(SC_LOGIC_0);\n\t\t\telse if (testVecs[testVecidx]["+str(s)+"] == '1')\n\t\t\t\ttestData"+str(s+1)+".write(SC_LOGIC_1);\n\n")
sysC_TB_C.append("\t\twait(15000, SC_NS);\n\t\tnewTV.write(SC_LOGIC_1);\n\t\twait(0, SC_NS);\n\t\t}\n\t\twait();\n\t}\n}\n")

sysC_TB_C.append("std::vector<std::string> "+benchmark_name+"_TB::read_testPtr(std::string filename)\n{\n")
sysC_TB_C.append("\tstd::vector<std::string> selected_testVec;\n\tstd::ifstream file(filename);\n\tif (!file.is_open()) {\n\t\tstd::cerr << \"Error opening file!\" << std::endl;\t\n\t}\n\tstd::string line;\n\tstd::vector<std::string> lines;\n\twhile (std::getline (file, line))\n\t{\n\t\tlines.push_back(line);\n\t}\n\treturn lines;\n}")

sysC_TB_C.append("\nvoid "+benchmark_name+"_TB::endOfSim()\n{\n\twhile (true)\n{\n\t\tend.write(SC_LOGIC_0);\n\t\twait(3900000, SC_NS);\n\t\tend.write(SC_LOGIC_1);\n\t\twait(15000, SC_NS);\n\t\twait();\n\t}\n}")

directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_C_output_file_path = os.path.join(directory_path+"TB.cpp") 
with open(sysC_TB_C_output_file_path, "w") as file:
    file.writelines(sysC_TB_C)
file.close()

In [19]:
sysC_sim_C = []
sysC_sim_C.append("#include \"TB.h\"\n#include <iostream>\n#include <fstream> \n\nint sc_main(int argc, char** argv)\n{\n\t"+benchmark_name+"_TB* TOP = new "+benchmark_name+"_TB(\"netlistSimulationTB_instance\");")
sysC_sim_C.append("\n\tsc_start(400000, SC_NS);\n\treturn 0;\n}")

directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_sim_C_output_file_path = os.path.join(directory_path+"simulation.cpp") 
with open(sysC_sim_C_output_file_path, "w") as file:
    file.writelines(sysC_sim_C)
file.close()